#**深度学习单轮训练完整逻辑与执行流程总结**  
  
  ##**1.张量定义与模型实例化**  
  张量（Tensor）作为深度学习的基础数据载体，是对多维数值矩阵的通用封装，同时内置自动求导计算图支持；首先完成输入特征张量、标签目标张量的构建，随后实例化线性全连接神经网络层（模型主体），该层内部自动初始化可学习权重矩阵与偏置向量，同时开启参数的自动梯度追踪（requires_grad=True），为后续参数优化提供前提条件。
  
  ##**2.前向传播(Forward Propagation)**  
  将输入张量传入已定义的神经网络模型，模型依据当前固定的权重与偏置参数，完成线性矩阵运算，输出本次推理的预测结果张量，该过程即为前向传播，本质是利用现有模型参数完成一次特征变换与结果预测。  
    
  ##**3.损失函数计算（Loss Calculation)**  
  引入预设损失函数（本次为均方误差 MSELoss，回归任务标准误差度量函数），接收模型预测输出与真实目标标签，计算二者之间的数值偏差，最终得到标量形式的损失值 loss。该损失值量化了当前模型单次推理的全局误差大小，损失数值越高，代表模型预测结果与真实分布偏差越大。  
    
  ##**4.梯度清零（Gradient Zeroing）**  
  调用优化器的zero_grad()方法，清空模型所有可学习参数上留存的上一轮迭代的历史梯度累积值。PyTorch 框架默认会对梯度进行累加复用，若不清零，本轮新计算的梯度会与历史梯度叠加，导致梯度更新方向偏移、模型训练发散无法收敛，因此单次迭代开始前的梯度重置是训练稳定的必要前置步骤。  
    
  ##**5.反向传播（Backward Propagation）**  
  对标量损失张量调用loss.backward()方法，启动反向梯度传播。该过程依托 PyTorch 内置的自动微分引擎，从损失值末端反向遍历整个计算图，利用链式求导法则，逐层计算损失函数对模型每一个可学习权重、偏置参数的偏导数，也就是梯度（Gradient），并将计算得到的梯度数值，自动写入每个模型参数的grad属性中。此步骤仅完成梯度的推导与存储，不会对模型本身的权重参数做任何修改。  
    
  ##**6.参数迭代更新（Parameter Update）**
  调用优化器的step()方法，依托上一步反向传播得到的全部参数梯度，结合预设的学习率（lr，梯度更新步长超参数），按照选定优化器（本次为 SGD 随机梯度下降算法）的更新规则，对模型所有权重与偏置参数进行数值微调，沿着损失降低的方向完成参数迭代，真正完成模型的 “学习” 过程。  
    
  ##**7.迭代循环与模型收敛**  
  将上述前向推理→损失计算→梯度清零→反向传播→参数更新的完整步骤封装为循环体，进行多轮迭代训练。随着迭代次数增加，损失值会持续稳步下降，模型预测输出与真实标签的偏差不断缩小，模型逐步拟合输入与输出之间的映射规律，完成整体的训练收敛。  
  
  ##**8.结果导出**  
  训练完成后，调用detach()将预测张量从自动求导计算图中剥离，阻断后续梯度追踪；再调用numpy()将 PyTorch 专属张量格式转换为通用 Numpy 数组格式，脱离框架依赖，用于最终结果的可视化、对比与落地使用。






In [1]:
import torch
a = torch.tensor(2)
b = torch.tensor(3)
c = a + b
print(f"a的值：{a}")
print(f"b的值：{b}")

#张量可以自动跟踪计算梯度;需要记录梯度，方便后面反向传播改错
a = torch.tensor(2.0, requires_grad=True)
result = a * 5

#反向传播，模拟AI"学习改错"
result.backward()
print(f"\nAI算出来的梯度：{a.grad}")


a的值：2
b的值：3

AI算出来的梯度：5.0


In [2]:
import torch

# 1维张量（类似列表）
vector = torch.tensor([1, 2, 3, 4])
print("1维张量：", vector)
print("维度张量：", vector.ndim)
print("形状：", vector.shape)

#2维张量 = 矩阵
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
    ])
print("\n2维张量：")
print(matrix)
print("维度数量：", matrix.ndim)
print("形状：", matrix.shape)

1维张量： tensor([1, 2, 3, 4])
维度张量： 1
形状： torch.Size([4])

2维张量：
tensor([[1, 2, 3],
        [4, 5, 6]])
维度数量： 2
形状： torch.Size([2, 3])


In [3]:
import torch

#3维张量(对应音乐/序列数据)
three_dim_tensor = torch.tensor([
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
])

print("3维张量：")
print(three_dim_tensor)

print("维度 ndim: ", three_dim_tensor.ndim)
print("形状 shape: ", three_dim_tensor.shape)




3维张量：
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
维度 ndim:  3
形状 shape:  torch.Size([2, 2, 2])


In [4]:
import torch


#torch.rand(行数, 列数)
#意思：生成0~1之间的随机小数张量
random_tensor = torch.rand(2, 3)


print("随机张量：")
print(random_tensor)

print("\n维度 ndim：", random_tensor.ndim)
print("形状 shape：", random_tensor.shape)


随机张量：
tensor([[0.5035, 0.4427, 0.8271],
        [0.9032, 0.1673, 0.8374]])

维度 ndim： 2
形状 shape： torch.Size([2, 3])


In [2]:
integer = torch.tensor(1234)
decimal = torch.tensor(3.1415926)
print("整数0维张量：", integer)
print("小数0维张量：", decimal)

整数0维张量： tensor(1234)
小数0维张量： tensor(3.1416)


In [ ]:
import torch
import torch.nn as nn # 神经网络工具箱

#1. 创建一个简单的AI大脑(线性层)
# 输入3个数字 --> 输出2个数字
model = nn.Linear(in_features=3, out_features=2)

#2.看看AI一开始的权重;
# weight = 权重 = AI的大脑参数
print("AI 初始权重(随机数)：")
print(model.weight)
print("权重形状：", model.weight.shape)

#3.准备输入数据+我们期待AI算出的正确答案
# 输入：1组、3个数字
x = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[0.5, 1.0]])

#4.AI第一次计算(前向传播)
prediction = model(x)
print("\nAI第一次瞎猜的结果：")
print(prediction)

#5.计算：AI这次猜错了多少
#MSELoss = Mean Squared Error = 均方误差
loss_fn = nn.MSELoss() # 基础误差计算器
loss = loss_fn(prediction, target)
print("\n这次猜错的总误差：", loss.item())

#6.optimizer = 优化器 = 专门帮AI改大脑参数，让它变聪明
#SGD(一种最常用的改错方法) = 随机梯度下降
optimizer = torch.optim.SGD(
    model.parameters(),  #模型参数
    lr=0.01  #学习率：每次
)

#7.开始改错流程
#grad = gradient = 梯度/把优化器里的梯度归零
optimizer.zero_grad()  #清空上一次的改错痕迹
loss.backward()  #反向传播，计算梯度/根据误差，反向计算出每个权重该怎么改

#8.修改权重第一步
optimizer.step()

#9.修改完权重后，重新算一次
new_prediction = model(x)
new_loss = loss_fn(new_prediction, target)

print("\n AI学习1次后的新结果：")
print(new_prediction)
print("改错后的新误差：", new_loss.item())



AI 初始权重(随机数)：
Parameter containing:
tensor([[ 0.3566,  0.4222, -0.0311],
        [-0.5592, -0.1555, -0.4776]], requires_grad=True)
权重形状： torch.Size([2, 3])

AI第一次瞎猜的结果：
tensor([[ 1.1695, -1.7276]], grad_fn=<AddmmBackward0>)

这次猜错的总误差： 3.944023370742798

 AI学习1次后的新结果：
tensor([[ 1.0690, -1.3185]], grad_fn=<AddmmBackward0>)
改错后的新误差： 2.849557638168335


In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(in_features=3, out_features=2)

x = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[0.5, 1.0]])

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)
print("开始训练！\n")

for step in range(100):
    prediction = model(x)
    loss = loss_fn(prediction, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #每10次打印一次成绩
    if step % 10 == 0:
        print(f"第{step:2}次训练 | 误差 = {loss.item():.4f}")

#------------------------------
#最后看结果
#------------------------------
print("\n训练完成！")
print("AI最终输出：", prediction.detach().numpy())
#detach:接下来我只是想看数字，不用再给这个数记录梯度、不用参与之后的反向传播更新权重了
#.numpy:把PyTorch专属张量，转成Python通用的普通数组
print("正确答案：      ", target.numpy())


    

    

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt